<a href="https://colab.research.google.com/github/DQ0115/1142programing_language/blob/main/HW2_%E6%88%90%E7%B8%BE%E4%B8%80%E6%9C%AC%E9%80%9A_part1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q google-generativeai

In [2]:
import gradio as gr
import pandas as pd
from google.colab import auth
from google.auth import default

# -*- coding: utf-8 -*-
import gspread
from datetime import datetime
import google.generativeai as genai
import os
import json

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [3]:
from google.colab import userdata
from google import genai

# 從 Colab Secrets 中獲取 API 金鑰
api_key = userdata.get('gemini')

# 使用獲取的金鑰配置 genai
client = genai.Client(api_key=api_key)

MODEL_ID = 'gemini-2.5-flash'

In [6]:
response = client.models.generate_content(
    model = MODEL_ID, contents="Explain how AI works in a few words"
)
print(response.text)

AI learns patterns from data to make predictions or decisions.


In [7]:
SHEET_URL = "https://docs.google.com/spreadsheets/d/1jR3qRQr2ZvWYKNuv8wen_-eTZWdc5a-LLvH7iymn2zw/edit?usp=sharing"
WORKSHEET_NAME = "工作表2"

REQUIRED_COLUMNS = ["日期", "科目", "作業成績"]

_auth_done = False
_gc = None
_ws = None

In [8]:
# --- 主要功能區塊 ---
def get_user_grades():
    """
    透過終端機輸入學生成績，直到使用者輸入 'q' 結束。
    """
    print("--- 準備輸入成績。輸入 'q' 來停止。---")
    grades = []
    while True:
        subject = input("請輸入科目（或輸入 'q' 停止）：")
        if subject.lower() == 'q':
            break

        grade = input(f"請輸入 {subject} 的成績：")
        try:
            grade = int(grade)
        except ValueError:
            print("成績必須是數字。請重新輸入。")
            continue

        today = datetime.now().strftime('%Y-%m-%d')
        grades.append([today, subject, grade])
        print(f"已記錄：日期: {today}, 科目: {subject}, 成績: {grade}\n")

    return grades

In [9]:
def get_ai_summary(grades):
    """
    呼叫 Gemini 模型來生成成績摘要與常見迷思。
    """
    # 準備給 AI 的提示
    prompt_text = "以下是學生的成績列表，請幫我根據這些成績，產出一個簡單的摘要與常見迷思整理（不評分，只做總結）。\n\n"
    for record in grades:
        date, subject, grade = record
        prompt_text += f"日期：{date}, 科目：{subject}, 成績：{grade}\n"

    print("\n--- 正在呼叫 AI 模型生成摘要... ---")
    try:
        response = client.models.generate_content(model = MODEL_ID, contents = prompt_text)
        summary = response.text
        return summary
    except Exception as e:
        print(f"呼叫 AI 時發生錯誤：{e}")
        return "AI 摘要生成失敗。"

In [10]:
new_grades = get_user_grades()

--- 準備輸入成績。輸入 'q' 來停止。---
請輸入科目（或輸入 'q' 停止）：國文
請輸入 國文 的成績：100
已記錄：日期: 2026-04-22, 科目: 國文, 成績: 100

請輸入科目（或輸入 'q' 停止）：數學
請輸入 數學 的成績：85
已記錄：日期: 2026-04-22, 科目: 數學, 成績: 85

請輸入科目（或輸入 'q' 停止）：自然
請輸入 自然 的成績：75
已記錄：日期: 2026-04-22, 科目: 自然, 成績: 75

請輸入科目（或輸入 'q' 停止）：q


In [11]:
new_grades

[['2026-04-22', '國文', 100], ['2026-04-22', '數學', 85], ['2026-04-22', '自然', 75]]

In [12]:
get_ai_summary(new_grades)


--- 正在呼叫 AI 模型生成摘要... ---


'好的，這是一份基於您提供的成績資料所做的摘要與常見迷思整理，全程不進行任何評價或判斷。\n\n---\n\n### 學生成績摘要\n\n*   **日期：** 2026年4月22日\n*   **科目與成績：**\n    *   國文：100分\n    *   數學：85分\n    *   自然：75分\n*   **整體觀察：**\n    本次成績記錄涵蓋國文、數學、自然三科。其中，國文科目獲得滿分100分；數學科目成績為85分；自然科目成績為75分。整體成績分佈在75分至100分之間。\n\n---\n\n### 成績評估的常見迷思整理\n\n在觀察學生成績時，人們常會無意中產生一些迷思，以下整理幾點常見的誤解，旨在提供一個更全面的視角，而非對學生進行評斷：\n\n1.  **迷思一：成績高低直接等同於學生能力好壞。**\n    *   **澄清：** 成績是特定時間點對特定學習內容的評量結果，它反映了學生在該次考試中對知識的掌握程度。但它受到多種因素影響，例如考試當下的身心狀態、題型熟悉度、學習策略、甚至試題難易度等。成績高不代表能力上限，成績低也不代表能力不足，它不能全面代表一個人的綜合能力、潛力或智力。\n\n2.  **迷思二：低分代表學生不努力或不聰明。**\n    *   **澄清：** 成績不佳可能有很多原因。除了缺乏努力外，也可能因為學習方法不適合、對某些概念理解有盲點、考試焦慮、健康狀況不佳，或是課程內容本身難度較高。將低分簡單歸因於「不努力」或「不聰明」過於武斷，忽略了背後可能存在的多元因素。\n\n3.  **迷思三：高分代表學生已經完全掌握所有知識。**\n    *   **澄清：** 即使獲得高分，也可能存在某些細節理解不夠深入，或者該次考試內容恰好是學生特別擅長的部分。高分固然值得肯定，但學習是持續的過程，仍可能存在進一步探究和提升的空間。\n\n4.  **迷思四：單次成績就能定論學生的學習傾向或未來發展。**\n    *   **澄清：** 一次成績僅為學習過程中的一個「快照」。學生的學習是一個動態且複雜的過程，需要觀察長期的成績趨勢、學習態度、參與度、興趣發展等多元面向，才能更全面地理解其學習狀況和潛力。\n\n5.  **迷思五：成績是學習唯一的、最重要的衡量標準。**\n    *   **澄清：** 成績

In [13]:
def main():
    """
    主程式流程：輸入成績 -> 獲取 AI 摘要 -> 寫入 Google Sheet。
    """
    try:
        # 1. Google Sheet 身份驗證
        auth.authenticate_user()

        creds, _ = default()
        gc = gspread.authorize(creds)

        sh = gc.open_by_url(SHEET_URL)
        ws = sh.worksheet(WORKSHEET_NAME)





        print("--- Google Sheet 連線成功。---")

        # 2. 獲取使用者輸入的成績
        new_grades = get_user_grades()

        if not new_grades:
            print("沒有輸入任何成績，程式結束。")
            return

        # 3. 將新成績寫入 Google Sheet
        ws.append_rows(new_grades)
        print("\n--- 成績已成功寫入 Google Sheet。---")

        # 4. 獲取 AI 摘要並寫入 Google Sheet
        summary = get_ai_summary(new_grades)

        # 尋找第一行空白列
        next_row = len(ws.col_values(1)) + 1

        # 使用 update_cell() 方法逐一更新儲存格
        ws.update_cell(next_row, 1, datetime.now().strftime('%Y-%m-%d'))
        ws.update_cell(next_row, 2, 'AI 摘要')

        # 為了避免單元格內容過長，將摘要內容分成多行來寫入
        summary_lines = summary.split('\n')
        for i, line in enumerate(summary_lines):
            ws.update_cell(next_row + i, 3, line)

        print("\n--- AI 摘要已成功寫入 Google Sheet。---")
        print("以下是 AI 生成的摘要內容：")
        print("-" * 50)
        print(summary)
        print("-" * 50)

    except gspread.exceptions.APIError as e:
        print(f"Google Sheets API 錯誤：{e.response.text}")
        print("請確認：")
        print("1. 您的服務帳戶金鑰檔案正確且未過期。")
        print("2. 您已將服務帳戶的 Email 地址（在 JSON 檔案中）分享給 Google Sheet，並給予編輯權限。")
    except Exception as e:
        print(f"發生未預期的錯誤：{e}")

if __name__ == "__main__":
    main()

--- Google Sheet 連線成功。---
--- 準備輸入成績。輸入 'q' 來停止。---
請輸入科目（或輸入 'q' 停止）：國文
請輸入 國文 的成績：100
已記錄：日期: 2026-04-22, 科目: 國文, 成績: 100

請輸入科目（或輸入 'q' 停止）：數學
請輸入 數學 的成績：85
已記錄：日期: 2026-04-22, 科目: 數學, 成績: 85

請輸入科目（或輸入 'q' 停止）：
請輸入  的成績：自然
成績必須是數字。請重新輸入。
請輸入科目（或輸入 'q' 停止）：75
請輸入 75 的成績：
成績必須是數字。請重新輸入。
請輸入科目（或輸入 'q' 停止）：q

--- 成績已成功寫入 Google Sheet。---

--- 正在呼叫 AI 模型生成摘要... ---

--- AI 摘要已成功寫入 Google Sheet。---
以下是 AI 生成的摘要內容：
--------------------------------------------------
好的，這是一份根據您提供的學生國文與數學成績所做的簡單摘要與常見迷思整理，全程秉持不評分，只做客觀總結的原則。

---

### **學生學習表現摘要**

根據您提供的資料，該學生在2026年4月22日記錄了兩科成績：

*   **國文科目：** 獲得100分
*   **數學科目：** 獲得85分

**總結：**
學生在特定日期（2026年4月22日）取得了國文與數學兩個科目的成績。其中，國文科目獲得滿分100分，數學科目獲得85分。兩科目之間的分數有所差異。

---

### **關於成績總結的常見迷思整理**

當我們檢視學生成績時，常常會不自覺地產生一些預設立場或誤解。以下整理幾個常見的迷思，旨在提醒在解讀成績時應保持客觀與全面的視角：

1.  **迷思一：以單次或少量成績概括學生全貌**
    *   **說明：** 這份資料僅包含學生在單一日期、兩個科目的成績。這僅是學生學習旅程中的一個片段，不足以全面評估學生的整體學習能力、潛力、學習習慣或對科目的真實興趣。例如，不能單憑國文滿分就斷定學生「天生擅長文科」，也不能僅憑數學85分就認定學生「數學不好」。

2.  **迷思二：忽略成績背後的脈絡與情境**
    *   **